# Module B — 分組與階層式資料的進階機器學習

對應 `2-Grouped_Data_ML.pptx`。示範情境 A (MERF) 與情境 B (Lasso/PLS)，資料集：UCI auto_mpg（教學）與 NASA 電池預後（進階，真實取樣）。


## 0. 安裝與載入


📊 對應投影片 → [第 4 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=4)

In [ ]:
!pip install -q merf==1.0 scikit-learn pandas numpy matplotlib



### 資料集：UCI auto_mpg（真實）
來源：CMU StatLib／UCI ML Repository（公開）。約 392 列（去除馬力缺值後）。

| 欄位 | 意義 | 單位/範圍 |
|---|---|---|
| mpg | 油耗（目標） | miles/gal |
| cylinders / displacement / horsepower / weight / acceleration | 引擎與車身規格 | 數值 |
| model_year | 年份 | 70–82 |
| **origin** | **產地（分組欄位）** | 1=美 2=歐 3=日 |

**本節分析目標**：預測 mpg，並揭露『相同規格、不同產地→不同 mpg』的組別異質性。詳見 `DATA_Module_B.md`。


In [ ]:
import numpy as np, pandas as pd
cols=['mpg','cylinders','displacement','horsepower','weight','acceleration','model_year','origin']
url='https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
df=pd.read_csv(url, sep=r'\s+', names=cols+['car_name'], na_values='?')
df=df.dropna().reset_index(drop=True)
df.head()


### 分組欄位
`origin`（1=美國 2=歐洲 3=日本）作為名目組別。相同規格在不同產地群可有不同 MPG。


In [ ]:
df['origin']=df['origin'].astype(int)
print(df.groupby('origin')['mpg'].mean())


## 1. 驗證陷阱：隨機切分 vs 群組感知
示範 train_test_split 如何讓同一 origin 同時進訓練與測試，再用 GroupKFold 取得誠實的泛化估計。


📊 對應投影片 → [第 7 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=7)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
X=df[['cylinders','displacement','horsepower','weight','acceleration','model_year']]
y=df['mpg']; groups=df['origin']
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.2,random_state=42)
rf=RandomForestRegressor(random_state=42).fit(Xtr,ytr)
print('隨機切分 R2 (樂觀):', round(rf.score(Xte,yte),3))


In [ ]:
gkf=GroupKFold(n_splits=3)
scores=cross_val_score(RandomForestRegressor(random_state=42),X,y,groups=groups,cv=gkf,scoring='r2')
print('GroupKFold R2 (誠實):', scores.round(3), '平均', round(scores.mean(),3))


## 2. 情境 A：MERF
固定效果（隨機森林）學全體規格→油耗；隨機截距吸收 origin 群的隱藏差異。


📊 對應投影片 → [第 9 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=9)

In [ ]:
from merf import MERF
clusters=df['origin'].astype('category')
Z=np.ones((len(df),1))  # 隨機截距設計矩陣
idx=np.arange(len(df))
tr,te=train_test_split(idx,test_size=0.2,random_state=42)
mrf=MERF(max_iterations=10)
mrf.fit(X.iloc[tr], Z[tr], clusters.iloc[tr], y.iloc[tr])
print('各組學到的隨機截距 b:')
print(mrf.trained_b.round(3))


In [ ]:
yhat=mrf.predict(X.iloc[te], Z[te], clusters.iloc[te])
from sklearn.metrics import r2_score
print('MERF 測試 R2:', round(r2_score(y.iloc[te], yhat),3))


## 3. 情境 B：Lasso 與 PLS
Lasso 把冗餘特徵係數壓成 0；PLS 把高維 X 壓成少數與 y 共變最大的成分。


📊 對應投影片 → [第 14 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=14)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.cross_decomposition import PLSRegression
lasso=Pipeline([('s',StandardScaler()),('m',LassoCV(cv=5,random_state=42))]).fit(X,y)
print('Lasso 係數 (0 = 可省略):')
print(pd.Series(lasso.named_steps['m'].coef_, index=X.columns).round(3))


In [ ]:
pls=Pipeline([('s',StandardScaler()),('m',PLSRegression(n_components=3))]).fit(X,y)
print('PLS 壓到 3 個潛在成分，訓練 R2:', round(pls.score(X,y),3))


## 4. 進階：NASA 鋰電池壽命（真實取樣）
來源：NASA Ames PCoE Battery Data Set（18650 鋰電芯 B0005/6/7/18，充放電循環至壽命終止）。

| 欄位 | 意義 | 單位 |
|---|---|---|
| **Battery_ID** | **電芯（分組欄位）** | B0005… |
| cycle | 放電循環序 | — |
| mean_voltage / min_voltage | 放電電壓 | V |
| mean_temp / ambient_temp | 溫度 | °C |
| mean_current | 電流 | A |
| capacity | 該循環容量 | Ah |
| SOH | 健康狀態=capacity/初始 | 0–1 |

**本節分析目標**：以 Battery_ID 為組別、用 LeaveOneGroupOut 驗證對『未見過電芯』的 SOH 泛化。詳見 `DATA_Module_B.md`。


📊 對應投影片 → [第 23 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=23)

In [ ]:
nasa_url='https://raw.githubusercontent.com/yjjchen-nkust/course-lab/main/workshop_data/data/nasa_battery_sample.csv'
bat=pd.read_csv(nasa_url, comment='#')
print(bat['Battery_ID'].nunique(),'顆電芯', len(bat),'列'); bat.head()


In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
Xb=bat[['mean_voltage','min_voltage','mean_temp','mean_current','ambient_temp']]
yb=bat['SOH']; gb=bat['Battery_ID']
logo=LeaveOneGroupOut()
sc=cross_val_score(RandomForestRegressor(random_state=42),Xb,yb,groups=gb,cv=logo,scoring='r2')
print('LeaveOneGroupOut 各折 R2:', sc.round(2), '平均', round(sc.mean(),2))


## 5. 結業評量：架構選擇
**診斷情境**（先自己判斷 A/B 與方法，再看解答）：

1. 智慧電網：5000 戶用電，相同特徵卻用電差異大。
2. 半導體：500 感測器，多種校正同一良率，求最簡規則。
3. 農業：50 農場不同投入達同產量，求核心輪廓。
4. 電池驗證陷阱：train_test_split 得 99% 卻上線崩潰。


📊 對應投影片 → [第 26 頁](https://yjjchen-nkust.github.io/course-lab/workshop_data/2-Grouped_Data_ML.pdf#page=26)

**解答：**
1. 情境 A → MERF；Household_ID 為組別，隨機截距學各戶基線。
2. 情境 B → LassoCV；L1 把冗餘感測器係數壓成 0。
3. 情境 B → PLSRegression；壓成少數潛在耕作輪廓。
4. 隨機切分讓同一電池跨訓練/測試（洩漏）；改用 GroupKFold(groups=Battery_ID) 讓整顆電池留作測試。
